# 📈 Clinical Trial Enrollment Forecaster
### Forecasting Cancer Trial Trends with Google's TimesFM 2.5

This notebook analyzes historical clinical trial data from ClinicalTrials.gov and forecasts enrollment trends across cancer types using:
- **TimesFM 2.5** — Google Research's 200M parameter pretrained foundation model for time series
- **ARIMA** — Classical statistical baseline for comparison

**Cancer types analyzed:** Breast cancer, NSCLC, Colorectal cancer, ALL (Leukemia)

---

In [ ]:
# Install (run once)
# !pip install requests pandas numpy matplotlib statsmodels
# !pip install timesfm torch  # For TimesFM (optional, ~1GB download)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

from trial_fetcher import TrialFetcher
from analyzer import TrialAnalyzer
from forecaster import EnrollmentForecaster

fetcher = TrialFetcher()

## 1. Fetch Historical Trial Data

Pull trials across **all statuses** (completed, terminated, recruiting, etc.) to build a complete historical picture.

In [ ]:
cancer_types = [
    "breast cancer",
    "non-small cell lung cancer",
    "colorectal cancer",
    "acute lymphoblastic leukemia",
]

datasets = {}
for cancer in cancer_types:
    trials = fetcher.fetch_trials(condition=cancer, max_results=1000)
    datasets[cancer] = trials
    print(f"{cancer}: {len(trials)} trials fetched")

## 2. Landscape Analysis

For each cancer type: phase distribution, completion rates, enrollment stats, sponsor mix.

In [ ]:
analyzers = {}
timelines = {}

for cancer, trials in datasets.items():
    analyzer = TrialAnalyzer(trials, cancer_type=cancer)
    analyzer.print_summary()
    
    timeline = analyzer.build_monthly_timeline()
    analyzers[cancer] = analyzer
    timelines[cancer] = timeline
    
    chart = analyzer.plot_timeline(timeline)
    if chart:
        display(Image(filename=chart))

## 3. Enrollment Forecasting

### TimesFM 2.5
Google Research's foundation model for time series — a 200M parameter decoder-only model pretrained on diverse time-series data. Key features:
- Zero-shot forecasting (no fine-tuning needed)
- **Quantile forecasts** — produces uncertainty bands (10th-90th percentile), not just point estimates
- Supports up to 16k context length

### Why Quantile Forecasts Matter
In clinical development, decisions are made under uncertainty. Knowing that "we expect 8 new trials/month" is less useful than knowing "we expect 5-12 new trials/month with 80% confidence." This is the same principle behind prognostic covariate adjustment (PROCOVA) — using predicted outcomes **with uncertainty** to make better trial design decisions.

In [ ]:
all_results = {}

for cancer, timeline in timelines.items():
    if len(timeline) < 12:
        print(f"Skipping {cancer} — not enough data")
        continue
    
    forecaster = EnrollmentForecaster(timeline, cancer_type=cancer, forecast_months=12)
    results = forecaster.run()
    all_results[cancer] = results
    
    chart = forecaster.plot()
    if chart:
        display(Image(filename=chart))

## 4. Cross-Cancer Comparison

Compare forecasted enrollment rates across cancer types.

In [ ]:
print(f"\n{'Cancer Type':<35} {'Model':<20} {'Forecast (trials/mo)'}")
print("─" * 75)
for cancer, results in all_results.items():
    for model_name, data in results.items():
        print(f"{cancer:<35} {data.get('model', model_name):<20} {data['forecast_mean']:.1f}")

## 5. Deep Dive: Breast Cancer Forecast

Let's look at the breast cancer forecast in detail — including the quantile uncertainty bands from TimesFM.

In [ ]:
bc_timeline = timelines.get("breast cancer")
bc_results = all_results.get("breast cancer", {})

if bc_timeline is not None and bc_results:
    # Show the raw forecast numbers
    last_date = bc_timeline["month"].max()
    future = pd.date_range(start=last_date + pd.DateOffset(months=1), periods=12, freq="MS")
    
    forecast_df = pd.DataFrame({"month": future})
    for name, data in bc_results.items():
        forecast_df[data.get("model", name)] = data["forecast"]
        if "quantile_10" in data and data["quantile_10"]:
            forecast_df[f"{data.get('model', name)} (10th pctile)"] = data["quantile_10"]
            forecast_df[f"{data.get('model', name)} (90th pctile)"] = data["quantile_90"]
    
    print("Breast Cancer — 12-Month Forecast:")
    print(forecast_df.to_string(index=False))
else:
    print("No breast cancer data available")

---

## Connection to Digital Twin Approaches

This project uses a **foundation model** (TimesFM) to forecast clinical trial time series — which is conceptually aligned with how companies like [Unlearn AI](https://unlearn.ai) use **generative models** to forecast patient-level clinical outcomes:

| This Project | Unlearn's Digital Twins |
|---|---|
| TimesFM (pretrained foundation model) | Neural Boltzmann Machines (generative model) |
| Forecasts monthly enrollment trends | Forecasts individual patient health trajectories |
| Quantile forecasts for uncertainty | Prognostic scores with uncertainty (PROCOVA) |
| ClinicalTrials.gov historical data | Historical patient-level clinical trial data |
| Helps sponsors understand the trial landscape | Helps sponsors optimize trial design |

Both approaches apply the same core idea: **use pretrained models on historical clinical data to forecast future outcomes with quantified uncertainty.**